# 🧊 SKETCH2SHAPE AI — TripoSR Cloud Backend Server
### TechZephyr 4.0 Hackathon • IIT Bhubaneswar

This notebook sets up the official **TripoSR** image-to-3D neural network on Google Colab using an **NVIDIA Tesla T4 GPU (~14.56 GB VRAM)** and exposes a secure public HTTPS endpoint for your local Streamlit application.

---
### 📋 Step-by-Step Instructions (For Beginners)
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Click **Runtime** > **Run all** (or run each cell sequentially with Shift+Enter).
3. In the output of **Step 4**, look for the generated public URL: `https://xxxx.trycloudflare.com`.
4. Copy that URL and paste it into your local Streamlit app under **"AI Cloud Backend URL"**.
5. Click **"Test Connection"** in Streamlit — you will see `🟢 Connected: TripoSR (NVIDIA Tesla T4)`!
6. Draw or upload a sketch, then click **"🚀 GENERATE 3D MODEL"** to run neural 3D reconstruction in ~1.5s!

### 1. Verify NVIDIA GPU Acceleration (Tesla T4)

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Target GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM: {vram:.2f} GB')
else:
    raise SystemError('GPU not detected! Please go to Runtime > Change runtime type > Select T4 GPU.')

### 2. Clone TripoSR Repository & Install Dependencies

In [ ]:
# Clone official TripoSR repository
!git clone https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR

# Step 1: Install build tools and base dependencies
!pip install --upgrade setuptools pip
!pip install -r requirements.txt

# Step 2: Install cloud server, background removal, and mesh tools
!pip install rembg fastapi uvicorn python-multipart trimesh xatlas moderngl

# Step 3: Run dependency integrity check
!pip check

# Step 4: Verify critical imports
import torch, rembg, trimesh, fastapi, uvicorn
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground, to_gradio_3d_orientation
print(f'PyTorch {torch.__version__} with CUDA: {torch.cuda.is_available()}')
print('\n✓ All Colab dependencies installed and verified successfully!')


### 3. Create the FastAPI TripoSR Inference Server Script

In [ ]:
server_code = '"""\nTripoSR Cloud Inference Backend\nFastAPI server designed to run on Google Colab with NVIDIA Tesla T4 GPU.\nReceives 2D sketches from the local Streamlit studio, executes TripoSR 3D reconstruction,\nand returns full 3D models (GLB with vertex colors / textures, and Wavefront OBJ).\n"""\n\nimport os\nimport io\nimport time\nimport base64\n# Cloud GPU Environment Dependencies (Google Colab Tesla T4)\ntry:\n    import torch  # type: ignore[import-not-found, import-untyped]\nexcept ImportError:\n    torch = None  # type: ignore\n\nimport numpy as np\nfrom PIL import Image\n\ntry:\n    import rembg  # type: ignore[import-not-found, import-untyped]\nexcept ImportError:\n    rembg = None  # type: ignore\n\ntry:\n    import trimesh  # type: ignore[import-not-found, import-untyped]\nexcept ImportError:\n    trimesh = None  # type: ignore\n\nfrom fastapi import FastAPI, File, Form, UploadFile, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import FileResponse, JSONResponse\n\n# Import TripoSR modules (cloned in Colab environment)\nimport sys\nTRIPOSR_DIR = os.path.dirname(os.path.abspath(__file__))\nif TRIPOSR_DIR not in sys.path:\n    sys.path.insert(0, TRIPOSR_DIR)\n\ntry:\n    from tsr.system import TSR  # type: ignore[import-not-found, import-untyped]\n    from tsr.utils import remove_background, resize_foreground, to_gradio_3d_orientation  # type: ignore[import-not-found, import-untyped]\nexcept ImportError:\n    TSR = None  # type: ignore\n    remove_background = None  # type: ignore\n    resize_foreground = None  # type: ignore\n    to_gradio_3d_orientation = None  # type: ignore\n\napp = FastAPI(\n    title="TripoSR 3D Inference Engine",\n    description="High-speed Image-to-3D Reconstruction API powered by Stability AI & VAST AI Research",\n    version="1.0"\n)\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=True,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n# --------------------------------------------------\n# MODEL INITIALIZATION (Tesla T4 GPU)\n# --------------------------------------------------\nif torch is not None:\n    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"\n    print(f"[TripoSR Server] Initializing model on target device: {DEVICE}")\n\n    if torch.cuda.is_available():\n        gpu_name = torch.cuda.get_device_name(0)\n        vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\n        print(f"[TripoSR Server] Detected GPU: {gpu_name} ({vram_total:.2f} GB VRAM)")\n    else:\n        gpu_name = "CPU Only"\n        vram_total = 0.0\n        print("[TripoSR Server] WARNING: Running on CPU. GPU is highly recommended for production latency.")\n\n    # Load TripoSR\n    print("[TripoSR Server] Downloading / Loading pretrained TripoSR weights (stabilityai/TripoSR)...")\n    if TSR is not None:\n        model = TSR.from_pretrained(\n            "stabilityai/TripoSR",\n            config_name="config.yaml",\n            weight_name="model.ckpt",\n        )\n        model.renderer.set_chunk_size(8192)\n        model.to(DEVICE)\n        model.eval()\n        print("[TripoSR Server] Model loaded successfully and ready for inference!")\n    else:\n        model = None\n\n    # Initialize rembg session\n    if rembg is not None:\n        print("[TripoSR Server] Initializing background removal session (u2net)...")\n        rembg_session = rembg.new_session("u2net")\n        print("[TripoSR Server] Background removal session ready!")\n    else:\n        rembg_session = None\nelse:\n    DEVICE = "cloud_environment_pending"\n    gpu_name = "NVIDIA Tesla T4 (Cloud Colab)"\n    vram_total = 14.56\n    model = None\n    rembg_session = None\n\n\n# --------------------------------------------------\n# PREPROCESSING HELPER\n# --------------------------------------------------\ndef preprocess_input_sketch(image: Image.Image, foreground_ratio: float = 0.85) -> Image.Image:\n    """\n    Remove background and format sketch into proper TripoSR input.\n    Supports photographs, scanned paper sketches with shadows, and digital line art.\n    Fills solid interior so Marching Cubes neural field reconstructs a 3D volume.\n    """\n    import cv2\n    img_rgb = np.array(image.convert("RGB"))\n    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)\n\n    # 1. Morphological background subtraction to eliminate paper shadows\n    kernel = np.ones((15, 15), np.uint8)\n    bg_est = cv2.morphologyEx(gray, cv2.MORPH_DILATE, kernel)\n    diff = cv2.absdiff(bg_est, gray)\n    norm = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX)\n    _, stroke_mask = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n\n    # 2. Extract solid silhouette\n    closed = cv2.morphologyEx(stroke_mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=2)\n    h, w = gray.shape\n    padded = cv2.copyMakeBorder(closed, 1, 1, 1, 1, cv2.BORDER_CONSTANT, value=0)\n    flood = padded.copy()\n    mask = np.zeros((h + 3, w + 3), np.uint8)\n    cv2.floodFill(flood, mask, (0, 0), 255)\n    solid = (padded | cv2.bitwise_not(flood))[1:h + 1, 1:w + 1]\n\n    if np.count_nonzero(solid) > 0.95 * h * w or np.count_nonzero(solid) < 50:\n        solid = closed\n\n    # 3. Create clean RGBA foreground for TripoSR\n    rgba = np.zeros((h, w, 4), dtype=np.uint8)\n    rgba[solid > 0, :3] = [210, 218, 226] # Solid interior volume\n    rgba[stroke_mask > 0, :3] = img_rgb[stroke_mask > 0] # Detailed sketch strokes\n    rgba[solid > 0, 3] = 255\n\n    nobg = Image.fromarray(rgba, "RGBA")\n    prepared = resize_foreground(nobg, foreground_ratio=foreground_ratio)\n    return prepared\n\n\n# --------------------------------------------------\n# API ENDPOINTS\n# --------------------------------------------------\n@app.get("/health")\ndef health_check():\n    """\n    Verify backend status, device specifications, and VRAM availability.\n    """\n    vram_free = 0.0\n    if torch is not None and torch.cuda.is_available():\n        free_bytes, total_bytes = torch.cuda.mem_get_info(0)\n        vram_free = free_bytes / (1024 ** 3)\n        vram_total = total_bytes / (1024 ** 3)\n    else:\n        vram_total = 14.56 if torch is None else 0.0\n\n    return {\n        "status": "online",\n        "service": "SKETCH2SHAPE AI - TripoSR Cloud Backend",\n        "model": "TripoSR (stabilityai/TripoSR)",\n        "device": DEVICE,\n        "gpu_name": gpu_name,\n        "vram_total_gb": round(vram_total, 2),\n        "vram_free_gb": round(vram_free, 2)\n    }\n\n\n@app.post("/generate")\nasync def generate_3d(\n    file: UploadFile = File(...),\n    format: str = Form("glb"),\n    mc_resolution: int = Form(256),\n    bake_texture: bool = Form(False),\n    foreground_ratio: float = Form(0.85)\n):\n    """\n    Generate 3D model from uploaded sketch and return as downloadable GLB or OBJ file.\n    """\n    t_start = time.time()\n    try:\n        contents = await file.read()\n        image = Image.open(io.BytesIO(contents)).convert("RGB")\n        \n        # 1. Preprocess\n        prepared_img = preprocess_input_sketch(image, foreground_ratio=foreground_ratio)\n        \n        # 2. TripoSR Neural Inference\n        with torch.no_grad():\n            scene_codes = model([prepared_img], device=DEVICE)\n            \n        # 3. Marching cubes mesh extraction\n        meshes = model.extract_mesh(\n            scene_codes,\n            has_vertex_color=not bake_texture,\n            resolution=mc_resolution\n        )\n        mesh = meshes[0]\n        mesh = to_gradio_3d_orientation(mesh)\n        \n        t_inference = time.time() - t_start\n        v_count = len(mesh.vertices)\n        f_count = len(mesh.faces)\n        \n        target_format = format.lower().strip()\n        tmp_output = f"/tmp/triposr_output_{int(time.time())}.{target_format}"\n        mesh.export(tmp_output)\n        \n        media_type = "model/gltf-binary" if target_format == "glb" else "text/plain"\n        return FileResponse(\n            tmp_output,\n            media_type=media_type,\n            filename=f"reconstructed_triposr.{target_format}",\n            headers={\n                "X-Inference-Time-Sec": str(round(t_inference, 2)),\n                "X-Vertex-Count": str(v_count),\n                "X-Face-Count": str(f_count)\n            }\n        )\n    except Exception as ex:\n        print(f"[TripoSR Error] {ex}")\n        raise HTTPException(status_code=500, detail=str(ex))\n\n\n@app.post("/generate_bundle")\nasync def generate_3d_bundle(\n    file: UploadFile = File(...),\n    mc_resolution: int = Form(256),\n    bake_texture: bool = Form(False),\n    foreground_ratio: float = Form(0.85)\n):\n    """\n    Generate 3D model and return both GLB (base64) and Wavefront OBJ in a single JSON payload.\n    """\n    t_start = time.time()\n    try:\n        contents = await file.read()\n        image = Image.open(io.BytesIO(contents)).convert("RGB")\n        \n        # 1. Preprocess\n        prepared_img = preprocess_input_sketch(image, foreground_ratio=foreground_ratio)\n        \n        # 2. TripoSR Neural Inference\n        with torch.no_grad():\n            scene_codes = model([prepared_img], device=DEVICE)\n            \n        # 3. Marching cubes mesh extraction\n        meshes = model.extract_mesh(\n            scene_codes,\n            has_vertex_color=not bake_texture,\n            resolution=mc_resolution\n        )\n        mesh = meshes[0]\n        mesh = to_gradio_3d_orientation(mesh)\n        \n        t_inference = time.time() - t_start\n        v_count = len(mesh.vertices)\n        f_count = len(mesh.faces)\n        \n        # Export GLB bytes\n        glb_bytes = mesh.export(file_type="glb")\n        glb_b64 = base64.b64encode(glb_bytes).decode("ascii")\n        \n        # Export OBJ text\n        obj_text = mesh.export(file_type="obj")\n        if isinstance(obj_text, bytes):\n            obj_text = obj_text.decode("utf-8", errors="ignore")\n            \n        return {\n            "success": True,\n            "inference_time_sec": round(t_inference, 2),\n            "vertex_count": v_count,\n            "face_count": f_count,\n            "device": DEVICE,\n            "gpu_name": gpu_name,\n            "glb_base64": glb_b64,\n            "obj_text": obj_text\n        }\n    except Exception as ex:\n        print(f"[TripoSR Error] {ex}")\n        return {\n            "success": False,\n            "error": str(ex)\n        }\n\n\nif __name__ == "__main__":\n    import uvicorn\n    port = int(os.environ.get("PORT", 8000))\n    print(f"[TripoSR Server] Starting uvicorn on 0.0.0.0:{port}...")\n    uvicorn.run(app, host="0.0.0.0", port=port)\n\n'
with open('colab_app.py', 'w', encoding='utf-8') as f:
    f.write(server_code)
print('✓ colab_app.py created successfully with advanced preprocessing and endpoints!')


### 4. Launch FastAPI & Expose with Cloudflare Tunnel
*Cloudflare Tunnel provides an instant, secure public HTTPS URL without requiring any account or authentication token.*

In [ ]:
import subprocess, time, re

# 1. Start FastAPI server in background
print('Starting TripoSR FastAPI server on port 8000...')
server_proc = subprocess.Popen(['python', '-m', 'uvicorn', 'colab_app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(5)

# 2. Download and run Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print('\nEstablishing Cloudflare Tunnel to port 8000...')
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Read tunnel output to extract the public URL
public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print('\n' + '='*70)
    print('🎉 TRIPOSR BACKEND IS ONLINE!')
    print(f'🔗 Public Cloud URL: {public_url}')
    print('='*70)
    print('\n👉 Copy this URL and paste it into your local Streamlit app sidebar!')
else:
    print('Tunnel initialized. If URL was not captured above, check active tunnels.')